# 04 · MapMyCells cell type mapping

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`map_cell_types`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`mmc`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/mmc) → `sc-mmc` |

[MapMyCells](https://brain-map.org/bkp/analyze/mapmycells) ([`cell_type_mapper`](https://github.com/AllenInstitute/cell_type_mapper)) assigns each cell to a node at every level of an Allen Brain Cell Atlas taxonomy, by correlating its expression with each node's average profile over a set of marker genes.

- **Hierarchical mapping.** A cell is assigned to the best class, then the best child within it, and so on.
- **Bootstrapping.** At each level the assignment is repeated 100 times on random halves of the markers (`bootstrap_factor = 0.5`). The *bootstrapping probability* is the fraction of iterations that agree; the *correlation coefficient* is how well the cell matches the chosen node.
- **On-the-fly markers.** With no marker JSON (human), markers are selected from the precomputed stats for the genes actually present in the query (`OnTheFlyMapper`). The mouse workflow passes a fixed marker JSON instead (`FromSpecifiedMarkersRunner`).
- **Raw counts, all genes.** The step runs on the filtered object *before* normalization and HVG selection, so it can use any marker gene. Gene symbols are mapped to Ensembl IDs (`map_to_ensembl`) to match the reference.

The results CSV has one row per cell with, for every taxonomy level, the label, name, bootstrapping probability and correlation coefficient. Notebook 06 turns them into cell type labels.

This step is CPU-only and memory-hungry: the WDL gives it 16 threads and about 8× the input size + 100 GB of RAM.

**Inputs**
- `${cohort_id}.filtered.h5ad` (raw counts, all genes)
- MapMyCells precomputed stats `.h5` for the chosen taxonomy

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.mmc_otf_mapping.${mmc_taxonomy}.results.csv`
- `….extended_results.json`
- `….log.txt`

**Previous:** [03_filter](03_filter.ipynb) · **Next:** [05_process](05_process.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Which taxonomy did the release use?

The taxonomy is part of the output file name.

In [ ]:
for p in sorted(cfg.CURATED_COHORT_PATH.glob("*.results.csv")):
    print("release:", p.name)
print("this run:", cfg.MMC_TAXONOMY, "→", cfg.MMC_PRECOMPUTED_STATS[cfg.MMC_TAXONOMY].name)

## 2. Run `sc-mmc`

In [ ]:
mmc_prefix = cfg.out(f"mmc_otf_mapping.{cfg.MMC_TAXONOMY}")
cfg.run(
    "sc-mmc",
    "--adata-input", cfg.out("filtered.h5ad"),
    "--mmc-precomputed-stats", cfg.MMC_PRECOMPUTED_STATS[cfg.MMC_TAXONOMY],
    "--n-processors", cfg.MMC_N_PROCESSORS,
    "--output-prefix", mmc_prefix,
)

## 3. Read the results

The CSV starts with `#` header lines that record the taxonomy levels; `read_csv_results` parses them.

In [ ]:
from sc_tools.transcriptional_phenotype import read_csv_results

mmc_csv = f"{mmc_prefix}.results.csv"
results, levels = read_csv_results(mmc_csv)
print("taxonomy levels:", levels)
print(f"{len(results):,} cells mapped")
results.head()

In [ ]:
top = levels[0]
results[f"{top}_name"].value_counts().to_frame()

## 4. Mapping confidence

Notebook 06 keeps a label only when both the correlation and the bootstrapping probability are ≥ 0.5.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(levels), 2, figsize=(10, 3 * len(levels)), constrained_layout=True)
for row, level in zip(axes, levels):
    for ax, stat in zip(row, ["correlation_coefficient", "bootstrapping_probability"]):
        ax.hist(results[f"{level}_{stat}"], bins=50, color="#4878a8")
        ax.axvline(0.5, color="k", linestyle="--", linewidth=1)
        ax.set_title(f"{level}: {stat}")
plt.show()

**Previous:** [03_filter](03_filter.ipynb) · **Next:** [05_process](05_process.ipynb)